# **Weald Valley Libraries: building the loans database**

In [1]:
try:
    import faker
except ImportError:
    %pip install -q faker

In [2]:
import os, sqlite3, random
from datetime import date, timedelta
import numpy as np
import pandas as pd
from faker import Faker

SEED = 7702
rand = np.random.default_rng(SEED)
random.seed(SEED)
fk = Faker(['en_GB'])
fk.seed_instance(SEED)
fk_world = Faker(['en_US', 'fr_FR', 'de_DE', 'it_IT', 'en_IE'])
fk_world.seed_instance(SEED + 1)

DB_FILE = 'weald_libraries.db'
FIRST_LOAN_DAY = date(2023, 1, 1)
LAST_DAY = date(2026, 9, 30)

## Table definitions

Eight tables. I put the rules in the database itself so bad rows get rejected on insert rather than
cleaned up afterwards. Two tables have a primary key made of two columns (`title_authors` and
`reservations`) and `copies` has a two-column UNIQUE rule, because accession numbers restart at each branch.

In [3]:
DDL = """
PRAGMA foreign_keys = ON;

CREATE TABLE branches (
    branch_id        INTEGER PRIMARY KEY,
    branch_name      TEXT NOT NULL UNIQUE,
    town             TEXT NOT NULL,
    code             TEXT NOT NULL UNIQUE CHECK (code GLOB '[A-Z][A-Z]'),
    opened_year      INTEGER NOT NULL CHECK (opened_year BETWEEN 1850 AND 2026),
    floor_area_m2    REAL NOT NULL CHECK (floor_area_m2 > 0),
    weekly_hours     REAL NOT NULL CHECK (weekly_hours BETWEEN 0 AND 84),
    has_cafe         INTEGER NOT NULL DEFAULT 0 CHECK (has_cafe IN (0, 1))
);

CREATE TABLE members (
    member_id          INTEGER PRIMARY KEY,
    first_name         TEXT NOT NULL CHECK (length(first_name) > 0),
    last_name          TEXT NOT NULL CHECK (length(last_name) > 0),
    email              TEXT CHECK (email IS NULL OR (email LIKE '%@%' AND email NOT LIKE '@%' AND instr(email, '.') > 0)),
    phone              TEXT CHECK (phone IS NULL OR phone GLOB '07[0-9][0-9][0-9] [0-9][0-9][0-9] [0-9][0-9][0-9]'),
    postcode_district  TEXT NOT NULL CHECK (substr(postcode_district, 1, 2) IN ('ME', 'TN') AND length(postcode_district) IN (3, 4)),
    birth_year         INTEGER NOT NULL CHECK (birth_year BETWEEN 1920 AND 2022),
    membership_type    TEXT NOT NULL CHECK (membership_type IN ('Adult', 'Junior', 'Concession', 'Student')),
    home_branch_id     INTEGER NOT NULL REFERENCES branches (branch_id) ON DELETE RESTRICT,
    joined_on          TEXT NOT NULL CHECK (julianday(joined_on) IS NOT NULL AND joined_on GLOB '[12][0-9][0-9][0-9]-[01][0-9]-[0-3][0-9]')
);

CREATE TABLE authors (
    author_id    INTEGER PRIMARY KEY,
    first_name   TEXT NOT NULL,
    last_name    TEXT NOT NULL,
    birth_year   INTEGER CHECK (birth_year IS NULL OR birth_year BETWEEN 1750 AND 2005)
);

CREATE TABLE titles (
    isbn13            TEXT PRIMARY KEY CHECK (length(isbn13) = 13 AND isbn13 GLOB '97[89]*' AND isbn13 NOT GLOB '*[^0-9]*'),
    title             TEXT NOT NULL,
    genre             TEXT NOT NULL CHECK (genre IN ('Crime', 'General fiction', 'Romance', 'Science fiction', 'Fantasy',
                                                  'Biography', 'History', 'Cookery', 'Travel', 'Young adult',
                                                  'Children''s picture book', 'Children''s fiction')),
    age_rating        TEXT NOT NULL CHECK (age_rating IN ('All ages', '8+', '12+', '15+', 'Adult')),
    publication_year  INTEGER NOT NULL CHECK (publication_year BETWEEN 1800 AND 2026),
    page_count        INTEGER CHECK (page_count IS NULL OR (page_count >= 12 AND page_count <= 1600)),
    list_price_gbp    REAL NOT NULL CHECK (list_price_gbp > 0)
);

CREATE TABLE title_authors (
    isbn13        TEXT NOT NULL REFERENCES titles (isbn13) ON DELETE CASCADE,
    author_id     INTEGER NOT NULL REFERENCES authors (author_id) ON DELETE RESTRICT,
    author_order  INTEGER NOT NULL DEFAULT 1 CHECK (author_order BETWEEN 1 AND 5),
    PRIMARY KEY (isbn13, author_id)
);

CREATE TABLE copies (
    copy_id        INTEGER PRIMARY KEY,
    isbn13         TEXT NOT NULL REFERENCES titles (isbn13) ON DELETE RESTRICT,
    branch_id      INTEGER NOT NULL REFERENCES branches (branch_id) ON DELETE RESTRICT,
    accession_no   TEXT NOT NULL CHECK (accession_no GLOB '[A-Z][A-Z]-[0-9][0-9][0-9][0-9][0-9]'),
    acquired_on    TEXT NOT NULL CHECK (date(acquired_on) = acquired_on),
    condition      TEXT CHECK (condition IS NULL OR condition IN ('New', 'Good', 'Fair', 'Poor')),
    UNIQUE (branch_id, accession_no)
);

CREATE TABLE loans (
    loan_id       INTEGER PRIMARY KEY,
    copy_id       INTEGER NOT NULL REFERENCES copies (copy_id) ON DELETE CASCADE,
    member_id     INTEGER NOT NULL REFERENCES members (member_id) ON DELETE CASCADE,
    issued_on     TEXT NOT NULL CHECK (date(issued_on) = issued_on),
    due_on        TEXT NOT NULL CHECK (date(due_on) = due_on AND due_on > issued_on),
    returned_on   TEXT CHECK (returned_on IS NULL OR (date(returned_on) = returned_on AND returned_on >= issued_on)),
    renewals      INTEGER NOT NULL DEFAULT 0 CHECK (renewals BETWEEN 0 AND 3),
    fine_pence    INTEGER NOT NULL DEFAULT 0 CHECK (fine_pence BETWEEN 0 AND 1000)
);

CREATE TABLE reservations (
    member_id         INTEGER NOT NULL REFERENCES members (member_id) ON DELETE CASCADE,
    isbn13            TEXT NOT NULL REFERENCES titles (isbn13) ON DELETE CASCADE,
    pickup_branch_id  INTEGER NOT NULL REFERENCES branches (branch_id) ON DELETE RESTRICT,
    placed_on         TEXT NOT NULL CHECK (date(placed_on) = placed_on),
    status            TEXT NOT NULL CHECK (status IN ('Waiting', 'Ready', 'Collected', 'Expired', 'Cancelled')),
    PRIMARY KEY (member_id, isbn13)
);

CREATE INDEX ix_loans_copy ON loans (copy_id, issued_on);
CREATE INDEX ix_loans_member ON loans (member_id);
"""

## Lookup lists

These are typed out by hand. The branch towns are real places in west Kent but the libraries, sizes and
opening hours are invented.

In [4]:
BRANCH_LIST = [
    # name, town, code, opened, m2, hours, cafe
    ('Maidstone Central Library', 'Maidstone', 'MA', 1894, 2150.0, 60.0, 1),
    ('Tonbridge Library', 'Tonbridge', 'TO', 1903, 980.0, 52.5, 1),
    ('Sevenoaks Library', 'Sevenoaks', 'SE', 1986, 1240.0, 54.0, 1),
    ('Tunbridge Wells Library', 'Royal Tunbridge Wells', 'TW', 1939, 1460.0, 56.0, 0),
    ('Paddock Wood Library', 'Paddock Wood', 'PW', 2004, 310.0, 31.5, 0),
    ('Edenbridge Library', 'Edenbridge', 'ED', 1972, 260.0, 28.0, 0),
]
POSTCODES = {1: ['ME14', 'ME15', 'ME16', 'ME17'], 2: ['TN9', 'TN10', 'TN11'], 3: ['TN13', 'TN14', 'TN15'],
             4: ['TN1', 'TN2', 'TN4'], 5: ['TN12'], 6: ['TN8']}

# genre: share of stock, mean pages, sd pages, age rating, loan pull (how often it goes out)
GENRES = {
    'Crime': (0.15, 380, 70, 'Adult', 1.4),
    'General fiction': (0.16, 340, 80, '15+', 1.0),
    'Romance': (0.07, 330, 60, '15+', 1.1),
    'Science fiction': (0.05, 420, 110, '12+', 0.8),
    'Fantasy': (0.05, 480, 130, '12+', 0.9),
    'Biography': (0.07, 360, 90, 'Adult', 0.6),
    'History': (0.07, 410, 120, 'Adult', 0.5),
    'Cookery': (0.04, 260, 50, 'All ages', 0.4),
    'Travel': (0.04, 300, 70, 'All ages', 0.3),
    'Young adult': (0.08, 360, 70, '12+', 1.0),
    "Children's picture book": (0.11, 32, 4, 'All ages', 1.6),
    "Children's fiction": (0.11, 190, 50, '8+', 1.3),
}

WORDS_A = ['Silent', 'Last', 'Hidden', 'Broken', 'Golden', 'Winter', 'Quiet', 'Lost', 'Little', 'Burning', 'Northern',
           'Secret', 'Long', 'Glass', 'Paper', 'Distant', 'Wild', 'Salt', 'Night', 'Crooked']
WORDS_B = ['Harbour', 'Garden', 'Orchard', 'House', 'River', 'Lighthouse', 'Letters', 'Shore', 'Field', 'Fox', 'Bridge',
           'Daughter', 'Summer', 'Island', 'Road', 'Kingdom', 'Promise', 'Witness', 'Kitchen', 'Map']
PLACES = ['Kent', 'Venice', 'the Marsh', 'Cornwall', 'Kyoto', 'the Downs', 'Lisbon', 'Skye', 'the Weald', 'Hastings']

## Branches and members

Member type drives the birth year, and the email is made from the member's own name.

In [5]:
branch_rows = []
for i, b in enumerate(BRANCH_LIST, start=1):
    branch_rows.append(dict(branch_id=i, branch_name=b[0], town=b[1], code=b[2], opened_year=b[3],
                            floor_area_m2=b[4], weekly_hours=b[5], has_cafe=b[6]))
branch_weight = np.array([b['floor_area_m2'] for b in branch_rows])
branch_weight = branch_weight / branch_weight.sum()

TYPE_YEARS = {'Adult': (1958, 2005), 'Junior': (2009, 2021), 'Concession': (1932, 1964), 'Student': (1998, 2008)}
providers = ['gmail.com', 'hotmail.co.uk', 'outlook.com', 'yahoo.co.uk', 'btinternet.com', 'icloud.com', 'sky.com']

def make_email(first, last):
    f = ''.join(c for c in first.lower() if c.isalpha())
    l = ''.join(c for c in last.lower() if c.isalpha())
    pick = rand.integers(0, 4)
    if pick == 0: user = f + '.' + l
    elif pick == 1: user = f[0] + l + str(rand.integers(1, 100))
    elif pick == 2: user = f + l
    else: user = l + '.' + f + str(rand.integers(60, 100))
    return user + '@' + rand.choice(providers, p=[.36, .18, .15, .11, .09, .07, .04])

member_rows = []
for mid in range(1, 2001):
    kind = rand.choice(['Adult', 'Junior', 'Concession', 'Student'], p=[.6, .19, .15, .06])
    lo, hi = TYPE_YEARS[kind]
    born = int(rand.integers(lo, hi + 1))
    home = int(rand.choice(range(1, 7), p=branch_weight))
    earliest = date(max(born + 4, 2004), 1, 1)
    joined = LAST_DAY - timedelta(days=int(rand.exponential(1500)))
    if joined < earliest:
        joined = earliest + timedelta(days=int(rand.integers(0, 400)))
    first = fk.first_name()
    last = fk.last_name()
    member_rows.append(dict(member_id=mid, first_name=first, last_name=last, email=make_email(first, last),
                            phone='07%d %03d %03d' % (rand.integers(100, 1000), rand.integers(0, 1000), rand.integers(0, 1000)),
                            postcode_district=str(rand.choice(POSTCODES[home])), birth_year=born,
                            membership_type=str(kind), home_branch_id=home, joined_on=min(joined, LAST_DAY).isoformat()))
print(len(member_rows), 'members')
pd.Series([m['membership_type'] for m in member_rows]).value_counts()

2000 members


Adult         1207
Junior         366
Concession     311
Student        116
Name: count, dtype: int64

## Authors, titles and who wrote what

ISBNs get a proper check digit. A title has one author most of the time, sometimes two or three.

In [6]:
def isbn_with_check(body12):
    total = sum(int(d) * (1 if i % 2 == 0 else 3) for i, d in enumerate(body12))
    return body12 + str((10 - total % 10) % 10)

author_rows = []
for aid in range(1, 461):
    src = fk if rand.random() < 0.7 else fk_world
    author_rows.append(dict(author_id=aid, first_name=src.first_name(), last_name=src.last_name(),
                            birth_year=int(np.clip(rand.normal(1965, 18), 1800, 2000))))

genre_names = list(GENRES)
genre_p = np.array([GENRES[g][0] for g in genre_names]); genre_p = genre_p / genre_p.sum()

def make_title(genre):
    style = rand.integers(0, 4)
    if style == 0: t = 'The ' + rand.choice(WORDS_A) + ' ' + rand.choice(WORDS_B)
    elif style == 1: t = 'The ' + rand.choice(WORDS_B) + ' of ' + rand.choice(PLACES)
    elif style == 2: t = rand.choice(WORDS_A) + ' ' + rand.choice(WORDS_B)
    else: t = 'A ' + rand.choice(WORDS_B) + ' for ' + fk.first_name()
    if genre == 'Cookery': t = rand.choice(['Simple', 'Weeknight', 'Seasonal', 'One Pan', 'Slow']) + ' ' + rand.choice(['Suppers', 'Baking', 'Kitchen', 'Vegetables', 'Puddings'])
    if genre == 'Travel': t = rand.choice(['Walking', 'Rail Journeys in', 'A Year in', 'Slow Travel in']) + ' ' + rand.choice(PLACES)
    return t

title_rows, ta_rows, seen_isbn = [], [], set()
for _ in range(1350):
    isbn = isbn_with_check('978' + ''.join(str(d) for d in rand.integers(0, 10, 9)))
    while isbn in seen_isbn:
        isbn = isbn_with_check('978' + ''.join(str(d) for d in rand.integers(0, 10, 9)))
    seen_isbn.add(isbn)
    g = str(rand.choice(genre_names, p=genre_p))
    share, mean_p, sd_p, rating, pull = GENRES[g]
    year = int(2026 - min(rand.exponential(9), 120)) if rand.random() > 0.06 else int(rand.integers(1850, 1960))
    pages = int(max(16, round(rand.normal(mean_p, sd_p))))
    if g == "Children's picture book": pages = int(rand.choice([24, 32, 32, 32, 40]))
    price = float(rand.choice([7.99, 8.99, 9.99, 12.99, 14.99, 16.99, 20.0, 22.0, 25.0],
                              p=[.18, .22, .2, .12, .1, .07, .05, .04, .02]))
    title_rows.append(dict(isbn13=isbn, title=make_title(g), genre=g, age_rating=rating,
                           publication_year=year, page_count=pages, list_price_gbp=price))
    n_auth = rand.choice([1, 2, 3], p=[.88, .1, .02])
    for order, aid in enumerate(rand.choice(range(1, 461), size=n_auth, replace=False), start=1):
        ta_rows.append(dict(isbn13=isbn, author_id=int(aid), author_order=order))

# how popular each title is: a few very popular, most rarely borrowed
popularity = {t['isbn13']: rand.pareto(1.6) * GENRES[t['genre']][4] + 0.05 for t in title_rows}
print(len(title_rows), 'titles,', len(ta_rows), 'author links')

1350 titles, 1546 author links


## Copies on the shelves

Popular titles get more copies, and bigger branches hold more stock. Accession numbers count up inside
each branch, which is why the UNIQUE rule is on (branch, accession number) together.

In [7]:
copy_rows = []
next_acc = {b: 1 for b in range(1, 7)}
for t in title_rows:
    n_copies = int(np.clip(1 + rand.poisson(popularity[t['isbn13']] * 0.9), 1, 8))
    earliest_buy = date(max(t['publication_year'], 2008), 1, 1)
    for _ in range(n_copies):
        br = int(rand.choice(range(1, 7), p=branch_weight))
        span = (LAST_DAY - earliest_buy).days
        bought = earliest_buy + timedelta(days=int(rand.integers(0, max(span, 1))))
        code = BRANCH_LIST[br - 1][2]
        copy_rows.append(dict(copy_id=len(copy_rows) + 1, isbn13=t['isbn13'], branch_id=br,
                              accession_no='%s-%05d' % (code, next_acc[br]), acquired_on=bought.isoformat(),
                              condition=None))
        next_acc[br] += 1
print(len(copy_rows), 'copies')

2869 copies


## Loan history

Each copy is walked through time on its own: it goes out, comes back some days later, sits on the shelf for
a while, then goes out again. Loans last 3 weeks (4 for juniors) and longer books are kept for longer. About two thirds of
borrowers renew when they need more time, up to three times. Anyone who does not renew and returns late builds up a fine of 25p a day for adults, half that
for concessions, and nothing for children. Books still out on 30 September 2026 have no return date.

In [8]:
members_by_branch = {b: [m for m in member_rows if m['home_branch_id'] == b] for b in range(1, 7)}
member_lookup = {m['member_id']: m for m in member_rows}
title_pages = {t['isbn13']: t['page_count'] for t in title_rows}
loan_rows = []

for c in copy_rows:
    pull = popularity[c['isbn13']]
    mean_gap = float(np.clip(420 / pull, 25, 900))
    day = max(date.fromisoformat(c['acquired_on']), FIRST_LOAN_DAY) + timedelta(days=int(rand.exponential(mean_gap)))
    times_out = 0
    while day <= LAST_DAY:
        pool = members_by_branch[c['branch_id']] if rand.random() < 0.85 else member_rows
        m = pool[int(rand.integers(0, len(pool)))]
        if date.fromisoformat(m['joined_on']) > day:
            day += timedelta(days=int(rand.integers(1, 10)))
            continue
        period = 28 if m['membership_type'] == 'Junior' else 21
        pages = title_pages[c['isbn13']]
        kept = int(np.clip(rand.gamma(2.2, (6 + pages / 45) / 2.2 * 1.6), 1, 150))
        needed = max(0, (kept - 1) // period)
        renew = min(3, needed) if rand.random() < 0.65 else 0   # about a third never renew and go overdue
        due = day + timedelta(days=period * (renew + 1))
        back = day + timedelta(days=kept)
        late_days = max(0, (back - due).days)
        rate = {'Adult': 25, 'Student': 25, 'Concession': 12, 'Junior': 0}[m['membership_type']]
        fine = min(late_days * rate, 1000)
        returned = back.isoformat() if back <= LAST_DAY else None
        if returned is None: fine = 0
        loan_rows.append(dict(loan_id=len(loan_rows) + 1, copy_id=c['copy_id'], member_id=m['member_id'],
                              issued_on=day.isoformat(), due_on=due.isoformat(), returned_on=returned,
                              renewals=int(renew), fine_pence=int(fine)))
        times_out += 1
        if returned is None: break
        day = back + timedelta(days=int(1 + rand.exponential(mean_gap)))
    # condition follows wear: more loans, worse state
    wear = times_out + rand.normal(0, 2)
    c['condition'] = 'New' if wear < 2 else 'Good' if wear < 9 else 'Fair' if wear < 18 else 'Poor'

print(len(loan_rows), 'loans')
pd.DataFrame(loan_rows)[['renewals', 'fine_pence']].describe().round(1)

14650 loans


,renewals,fine_pence
count,14650.0,14650.0
mean,0.3,30.5
std,0.6,125.3
min,0.0,0.0
25%,0.0,0.0
50%,0.0,0.0
75%,0.0,0.0
max,3.0,1000.0


## Reservations

People reserve the titles that are in demand. A member can only hold one reservation per title, so the key
is the pair (member, title).

In [9]:
hot = sorted(title_rows, key=lambda t: -popularity[t['isbn13']])[:260]
res_rows, taken = [], set()
while len(res_rows) < 1450:
    t = hot[int(min(rand.exponential(70), 259))]
    m = member_rows[int(rand.integers(0, len(member_rows)))]
    key = (m['member_id'], t['isbn13'])
    if key in taken: continue
    taken.add(key)
    placed = LAST_DAY - timedelta(days=int(rand.exponential(160)))
    placed = max(placed, date.fromisoformat(m['joined_on']))
    age = (LAST_DAY - placed).days
    if age < 14: status = rand.choice(['Waiting', 'Ready'], p=[.7, .3])
    else: status = rand.choice(['Collected', 'Expired', 'Cancelled'], p=[.78, .12, .1])
    res_rows.append(dict(member_id=m['member_id'], isbn13=t['isbn13'], pickup_branch_id=m['home_branch_id'] if rand.random() < .9 else int(rand.integers(1, 7)),
                         placed_on=placed.isoformat(), status=str(status)))
print(len(res_rows), 'reservations')

1450 reservations


## Mess added on purpose

A real library system is not clean, so I made it untidy in ways I have seen in real records. The rates are
set here and printed below, and the IDs are kept so they can be checked later.

| What | Where | How much | Why it happens |
|---|---|---|---|
| Missing | members.email | 3% | people who never gave one |
| Missing | members.phone | 6% | landline only or refused |
| Missing | authors.birth_year | 4% | not known for some writers |
| Missing | titles.page_count | 2% | catalogue record incomplete |
| Missing | copies.condition | 2% | not checked at the last stock take |
| Duplicate | members | 1.5% | person joined again at another branch, name typed slightly differently |
| Duplicate | loans | 0.6% | barcode scanned twice at the desk |

Books still on loan have no return date, but that is a real state of the data, not something I removed.

In [10]:
log = {}

def knock_out(rows, column, share, label):
    picked = rand.choice(len(rows), size=round(len(rows) * share), replace=False)
    for i in picked:
        rows[i][column] = None
    log[label] = len(picked)

knock_out(member_rows, 'email', 0.03, 'missing member email')
knock_out(member_rows, 'phone', 0.06, 'missing member phone')
knock_out(author_rows, 'birth_year', 0.04, 'missing author birth year')
knock_out(title_rows, 'page_count', 0.02, 'missing page count')
knock_out(copy_rows, 'condition', 0.02, 'missing copy condition')

# members who signed up twice
again = rand.choice(len(member_rows), size=round(len(member_rows) * 0.015), replace=False)
new_id = len(member_rows)
twins = []
for k, i in enumerate(again):
    m = dict(member_rows[i])
    new_id += 1
    m['member_id'] = new_id
    if k % 3 == 0: m['first_name'] = m['first_name'].upper()
    elif k % 3 == 1: m['last_name'] = ' ' + m['last_name']
    else: m['first_name'] = m['first_name'].lower()
    m['home_branch_id'] = int(rand.integers(1, 7))
    m['joined_on'] = min(date.fromisoformat(m['joined_on']) + timedelta(days=int(rand.integers(60, 700))), LAST_DAY).isoformat()
    twins.append(m)
member_rows.extend(twins)
log['duplicate members'] = len(twins)
log['originals of duplicated members'] = sorted(int(member_rows[i]['member_id']) for i in again)[:12]

# loans scanned twice
double = rand.choice(len(loan_rows), size=round(len(loan_rows) * 0.006), replace=False)
extra = []
for i in double:
    l = dict(loan_rows[i])
    l['loan_id'] = len(loan_rows) + len(extra) + 1
    extra.append(l)
loan_rows.extend(extra)
log['duplicate loans'] = len(extra)

for k, v in log.items():
    print(k.ljust(34), v)

missing member email               60
missing member phone               120
missing author birth year          18
missing page count                 27
missing copy condition             57
duplicate members                  30
originals of duplicated members    [11, 12, 121, 284, 285, 323, 364, 523, 531, 621, 718, 859]
duplicate loans                    88


## Into SQLite

Foreign keys are switched on first, so if any loan pointed at a copy or member that does not exist the
insert would fail.

In [11]:
if os.path.exists(DB_FILE):
    os.remove(DB_FILE)
db = sqlite3.connect(DB_FILE)
db.executescript(DDL)
db.execute('PRAGMA foreign_keys = ON')

def put(table, rows):
    cols = list(rows[0])
    sql = 'INSERT INTO %s (%s) VALUES (%s)' % (table, ', '.join(cols), ', '.join(':' + c for c in cols))
    db.executemany(sql, rows)

put('branches', branch_rows)
put('members', member_rows)
put('authors', author_rows)
put('titles', title_rows)
put('title_authors', ta_rows)
put('copies', copy_rows)
put('loans', loan_rows)
put('reservations', res_rows)
db.commit()

for (t,) in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"):
    print(t.ljust(16), db.execute('SELECT COUNT(*) FROM ' + t).fetchone()[0])

authors          460
branches         6
copies           2869
loans            14738
members          2030
reservations     1450
title_authors    1546
titles           1350


## Checks

First the built-in checks, then I look for my own mess to make sure it is all there.

In [12]:
print('integrity check:', db.execute('PRAGMA integrity_check').fetchone()[0])
print('foreign key problems:', db.execute('PRAGMA foreign_key_check').fetchall())
def sql(text):
    return pd.read_sql_query(text, db)
sql("""
SELECT 'member email blank' AS field, COUNT(*) FILTER (WHERE email IS NULL) AS missing FROM members
UNION ALL SELECT 'member phone blank', COUNT(*) FILTER (WHERE phone IS NULL) FROM members
UNION ALL SELECT 'author birth year blank', COUNT(*) FILTER (WHERE birth_year IS NULL) FROM authors
UNION ALL SELECT 'title page count blank', COUNT(*) FILTER (WHERE page_count IS NULL) FROM titles
UNION ALL SELECT 'copy condition blank', COUNT(*) FILTER (WHERE condition IS NULL) FROM copies
UNION ALL SELECT 'loans still out', SUM(returned_on IS NULL) FROM loans
""")

integrity check: ok
foreign key problems: []


,field,missing
0,member email blank,60
1,member phone blank,122
2,author birth year blank,18
3,title page count blank,27
4,copy condition blank,57
5,loans still out,321


In [13]:
sql("""
SELECT
  (SELECT COUNT(*) - COUNT(DISTINCT lower(trim(first_name)) || '|' || lower(trim(last_name)) || '|' || birth_year || '|' || IFNULL(phone, ''))
     FROM members) AS repeat_members,
  (SELECT COUNT(*) FROM loans) - (SELECT COUNT(*) FROM (SELECT DISTINCT copy_id, member_id, issued_on FROM loans)) AS repeat_loans
""")

,repeat_members,repeat_loans
0,30,88


## A few questions the data can answer

In [14]:
sql("""
SELECT t.genre, COUNT(*) AS loans, ROUND(AVG(julianday(l.returned_on) - julianday(l.issued_on)), 1) AS avg_days_kept
FROM loans l JOIN copies c ON c.copy_id = l.copy_id JOIN titles t ON t.isbn13 = c.isbn13
WHERE l.returned_on IS NOT NULL
GROUP BY t.genre ORDER BY loans DESC
""")

,genre,loans,avg_days_kept
0,Crime,3376,23.3
1,Children's picture book,2201,9.8
2,Children's fiction,2131,15.9
3,General fiction,2055,21.4
4,Romance,1367,20.4
5,Young adult,851,20.8
6,Fantasy,761,22.3
7,Biography,728,20.8
8,Cookery,303,18.4
9,History,265,18.0


In [15]:
sql("""
SELECT b.branch_name, COUNT(*) AS loans, ROUND(SUM(l.fine_pence) / 100.0, 2) AS fines_gbp
FROM loans l JOIN copies c USING (copy_id) JOIN branches b USING (branch_id)
GROUP BY b.branch_name ORDER BY loans DESC
""")

,branch_name,loans,fines_gbp
0,Maidstone Central Library,4996,1408.08
1,Tunbridge Wells Library,3619,1069.76
2,Sevenoaks Library,2862,1009.93
3,Tonbridge Library,2092,636.26
4,Edenbridge Library,590,179.39
5,Paddock Wood Library,579,183.17


In [16]:
db.close()
print('saved', DB_FILE)

saved weald_libraries.db
